# Intrusion Detection Model: RPL IoT Attack Classification

This notebook builds and compares three models, Logistic Regression, Random Forest, and
Gradient Boosting, to classify network traffic from a simulated IoT/RPL (Routing Protocol
for Low-power and Lossy Networks) dataset into 5 categories: Normal, Blackhole, Flooding,
Rank, and Version attacks.

The steps below go in order: load the raw data, clean it up, turn it into numbers a model
can understand, set up a baseline, then build, train, and compare each model.

## 1. Import Libraries

pandas and numpy handle data loading and array operations. scikit learn provides label
encoding, scaling, train and test splitting, and the four models used in this project.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load the Dataset

This CSV has a header row with real column names (from, to, frame_proto, protocol,
control_type, etc) describing simulated RPL (IoT routing protocol) traffic. The last
column, label, is always the target: Normal, Blackhole, Flooding, Rank, or Version.

In [ ]:
file_path = 'Dataset/RPL_Routing_Attacks.csv'

df = pd.read_csv(file_path, low_memory=False)

print('Rows, Columns:', df.shape)
df.head()

In [ ]:
label_col = df.columns[-1]
feature_cols = df.columns[:-1]

print('Label counts:')
print(df[label_col].value_counts())

## 3. Feature Selection: Removing the Leaking `from` Column

The from column (simulated device node ID) was found to fully determine the label: each
attacker node runs only one type of attack for the entire simulation (node 12 is always
Blackhole, node 14 is always Rank, and so on). This is data leakage, a model trained on
this column reaches 100% accuracy by memorizing node IDs instead of learning attack
behavior. The column was removed from the feature set for this reason.

In [ ]:
feature_cols = feature_cols.drop('from')

## 4. Data Cleaning and Feature Engineering

Each column was handled according to its content: comma separated lists (such as the
neighbor list in the to column) were converted to a neighbor count, columns that were at
least 90% numeric were kept as numbers with missing values filled as 0, and text or hex
coded columns (such as "RPL" or "AAAA0000") were label encoded into integer IDs.

In [ ]:
df[label_col] = df[label_col].astype(str)

processed = {}

for col in feature_cols:
    series = df[col]
    str_series = series.astype(str)

    comma_frac = str_series.str.contains(",", regex=False).mean()
    if comma_frac > 0.3:
        processed[col] = str_series.apply(lambda v: 0 if v in ("0", "nan") else len(v.split(",")))
        continue

    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().mean() > 0.9:
        processed[col] = numeric.fillna(0)
    else:
        processed[col] = LabelEncoder().fit_transform(str_series)

X = pd.DataFrame(processed)

X = X.replace([np.inf, -np.inf], 0).fillna(0)

print('Feature matrix shape:', X.shape)
X.head()

## 5. Label Encoding

Each label (Normal, Blackhole, Flooding, Rank, Version) was label encoded into an integer.
Logistic Regression, Random Forest, and Gradient Boosting all accept these integer labels
directly, with no further transformation needed.

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[label_col])
num_classes = len(label_encoder.classes_)

print('Classes found:', list(label_encoder.classes_))
print('Number of classes:', num_classes)

## 6. Feature Scaling

StandardScaler was applied to bring every feature to a comparable range, since raw values
ranged from simple 0/1 flags to protocol fields like DOAG_info reaching into the billions.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

num_samples = X_scaled.shape[0]
num_features = X_scaled.shape[1]
print('Samples:', num_samples, '| Features per sample:', num_features)

## 7. Baseline Model: Logistic Regression

Logistic Regression, a basic linear statistical classifier, was trained first to check
how much of the signal in the data is simple versus how much needs a stronger model to
capture, before moving on to Random Forest and Gradient Boosting.

In [ ]:
X_train_flat, X_test_flat, y_train_flat, y_test_flat = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

log_reg = LogisticRegression(max_iter=200, class_weight='balanced', n_jobs=-1)
log_reg.fit(X_train_flat, y_train_flat)
log_reg_preds = log_reg.predict(X_test_flat)
log_reg_acc = accuracy_score(y_test_flat, log_reg_preds)
print(f'Logistic Regression accuracy: {log_reg_acc:.4f}')
print()
print(classification_report(y_test_flat, log_reg_preds, target_names=label_encoder.classes_, digits=4))

## 8. Random Forest Classifier

The baselines above showed most of the signal is fairly simple, but Logistic Regression
still left real accuracy on the table. Random Forest is a stronger model built specifically
for tabular data like ours, using the same flat train test split from Section 7.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train_flat, y_train_flat)

rf_preds = rf.predict(X_test_flat)
rf_acc = accuracy_score(y_test_flat, rf_preds)

print(f'Random Forest Test Accuracy: {rf_acc:.4f}')

## 9. Random Forest: Classification Report and Confusion Matrix

A per class precision, recall, and confusion matrix breakdown was generated for the
Random Forest model, since overall accuracy alone does not show which specific attack
types are being confused with each other.

In [ ]:
print(classification_report(y_test_flat, rf_preds, target_names=label_encoder.classes_, digits=4))

cm = confusion_matrix(y_test_flat, rf_preds)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print('Confusion matrix (rows = actual, columns = predicted):')
print(cm_df)

## 10. Gradient Boosting Classifier

Gradient Boosting (HistGradientBoostingClassifier) was trained next, using the same flat
train test split, to test whether a sequential, error correcting ensemble could improve
on Random Forest's result.

In [ ]:
hgb = HistGradientBoostingClassifier(random_state=42, max_iter=200)
hgb.fit(X_train_flat, y_train_flat)

hgb_preds = hgb.predict(X_test_flat)
hgb_acc = accuracy_score(y_test_flat, hgb_preds)

print("Final model comparison:")
print(f"  Logistic Regression:  {log_reg_acc:.4f}")
print(f"  Random Forest:        {rf_acc:.4f}")
print(f"  Gradient Boosting:    {hgb_acc:.4f}")
print()
print(classification_report(y_test_flat, hgb_preds, target_names=label_encoder.classes_, digits=4))